# Processo de carga do proceso de Período de Experiência e atualização do PBI

Carregamento das bases, padronização de nomes de colunas e formatos de dados, além da atualização do PBI.

# Importação das Bibliotecas

In [ ]:
import pandas as pd
import duckdb as db
from datetime import datetime, date
from openpyxl import Workbook, load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.worksheet.table import Table, TableStyleInfo
import pyautogui
import pyperclip
import time
import os

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Processo de Importação finalizado')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Processo de Importação finalizado

----------------------------------------------------------------------------------------------------


# Carregando Base de Controle de Processos

In [28]:
# Carregamento da base de controle de processos

id = 1

path_registros_processos = r'X:\Gestao_de_Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\PROCESSOS.xlsx'

registros_processos = pd.read_excel(path_registros_processos, sheet_name = "REGISTROS", engine='openpyxl')

wb_p = load_workbook(path_registros_processos)

ws_p = wb_p['REGISTROS']

# Controle de atualização de processo: Etapa 0

linha_0 = [id, datetime.today(), 0]

ws_p.append(linha_0)

wb_p.save(path_registros_processos)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Registro de processos')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Registro de processos

----------------------------------------------------------------------------------------------------


# Carregamento de Bases

In [29]:
# Base de colaboradores

colaboradores = pd.read_excel(r'X:\Gestao_de_Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\COLABORADORES.xlsx')

# Base de afastamentos

afastamentos = pd.read_excel(r'X:\Gestao_de_Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\AFASTAMENTOS.xlsx')

# Base de unidades

path_unidades = r'X:\Gestao_de_Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\DIMENSÕES POWER BI.xlsx'

unidades = pd.read_excel(path_unidades, sheet_name='UNIDADES', usecols='B:F', skiprows=3)

# Controle de atualização de processo: Etapa 1

linha_1 = [id, datetime.today(), 1]

ws_p.append(linha_1)

wb_p.save(path_registros_processos)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Carregamento de bases concluído')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Carregamento de bases concluído

----------------------------------------------------------------------------------------------------


# Inicializando as Variáveis

In [30]:
# Filtrando apenas os colaboradores que foram admitidos a partir de 2024

colaboradores = db.query("""

    SELECT
            *

    FROM
            colaboradores

    WHERE
            data_admissao >= '2024-01-01'

""").df()

# Base de afastamentos excluindo afastamentos por férias

afastamentos_sem_ferias = db.query("""

    SELECT
            *
            
    FROM
            afastamentos

    WHERE
            descricao_afastamento <> 'Gozo de férias ou recesso - Afastamento temporário para o gozo de férias ou recesso'            

""").df()

# Variável com a data de afastamento mais antiga 

data_inicio_calendar = db.query("""

    SELECT 
            MIN( data_afastamento ) AS Data 
                
    FROM 
            afastamentos_sem_ferias
            
""").fetchone()[0].date()

# Variável com a maior data de retorno de afastamento 

data_fim_calendar = db.query("""

    SELECT 
            DATE_ADD(
                CASE 
                    WHEN CAST( MAX( data_retorno ) AS DATE ) < CAST( current_date AS DATE ) 
                    THEN CAST( current_date AS DATE ) 
                    ELSE CAST( MAX( data_retorno ) AS DATE )
                END,
                INTERVAL 90 DAY
            )   AS Dia
            
    FROM 
            afastamentos_sem_ferias
            
""").fetchone()[0]

# Controle de atualização de processo: Etapa 2

linha_2 = [id, datetime.today(), 2]

ws_p.append(linha_2)

wb_p.save(path_registros_processos)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Variáveis definidas')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Variáveis definidas

----------------------------------------------------------------------------------------------------


# Criação da Tabela de Dias e Registros

In [31]:
# Cariação da tabela calendário

db.sql(f"""

    CREATE OR REPLACE TABLE calendario AS
    
    SELECT * FROM generate_series(
        '{data_inicio_calendar}'::DATE,
        '{data_fim_calendar}'::DATE,
        INTERVAL '1 day'
    ) AS data;
    
""")

calendario = db.sql( "SELECT * FROM calendario" ).df()

# Base com os registros únicos de colaboradores

registros_unicos = db.query("""

    SELECT
            DISTINCT registro
        
    FROM
            colaboradores

""").df()

# Base com todas as datas e todos os registros

registro_vs_datas = db.query("""

    SELECT
            A.REGISTRO AS Registro,
            B.generate_series AS Data
                
    FROM
            registros_unicos A
        
    CROSS JOIN
            calendario B

""").df()

# Base com todas as datas e todos os registros, filtrando datas após admissão de cada colaborador

registro_vs_datas_consolidado = db.query("""

    SELECT
            A.Registro,
            A.Data

    FROM
            registro_vs_datas AS A

    INNER JOIN
            colaboradores AS B
        ON
            A.Registro = B.registro
        AND
            A.Data >= B.data_admissao

""").df()

# Controle de atualização de processo: Etapa 3

linha_3 = [id, datetime.today(), 3]

ws_p.append(linha_3)

wb_p.save(path_registros_processos)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Tabelas criadas')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Tabelas criadas

----------------------------------------------------------------------------------------------------


# Base com Marcação de Afastamento e Contagem de Dias

In [32]:
# Base com todos os registros e datas tendo flag se esteve afastado  

afastados_dia = db.query("""

    SELECT DISTINCT
            A.Registro,
            A.Data,
            CASE WHEN B.registro IS NOT NULL THEN 1 ELSE 0 END AS Afastado,
            CASE WHEN B.registro IS NOT NULL THEN 0 ELSE 1 END AS Apto

    FROM
            registro_vs_datas_consolidado A
            
    LEFT JOIN
            afastamentos_sem_ferias B
        ON
            A.Registro = B.REGISTRO
        AND
            A.Data BETWEEN B.data_afastamento AND CASE WHEN data_retorno IS NULL THEN '2050-12-31' ELSE data_retorno END 

""").df()

# Base com a criação do campo que agrupa momentos de afastamentos

contagem_acumulada_grupos = db.query("""

    SELECT 
        Registro,
        Data,
        Apto,
        Afastado,
        CAST( SUM( Apto ) 
            OVER (
                PARTITION BY Registro
                ORDER BY Data
                ROWS UNBOUNDED PRECEDING
            ) 
        AS INT ) AS Grupo
        
    FROM
        afastados_dia

""").df()

# Criação da contagem de dias acumulados e zerando quando apto ao trabalho

contagem_acumulada = db.query("""

    SELECT 
            Registro,
            Data,
            Apto,
            grupo,
            CAST( SUM(Afastado) OVER (
                PARTITION BY Registro, grupo
                ORDER BY Data
                ROWS UNBOUNDED PRECEDING
            ) AS INT ) AS Dias_afastados
        
    FROM
            contagem_acumulada_grupos

""").df()

# Base com aspenas os colaboradores que tiveram afastamentos acima de 15 dias

afastamentos_maior_15_dias = db.query("""

    WITH Afastados_caixa AS (
    
        SELECT
                Registro,
                Data,
                Grupo,
                Dias_afastados,
                ROW_NUMBER() OVER( PARTITION BY Registro, Grupo ORDER BY Dias_afastados DESC ) AS Rnk
    
        FROM
                contagem_acumulada
    
        WHERE
                Dias_afastados > 15
    
    )

    SELECT
            Registro,
            Data,
            Grupo,
            Dias_afastados

    FROM
            Afastados_caixa

    WHERE
            Rnk = 1

""").df()

# Base consolidada de colaboradores e datas com marcação de afastamento e se contabiliza para período de experiência

contabiliza_periodo_experiencia = db.query("""

    SELECT
            A.Registro,
            A.Data,
            A.Apto,
            A.Grupo,
            A.Dias_afastados,
            CASE WHEN B.Registro IS NULL THEN 1 ELSE 0 END AS Contabiliza,
            CAST ( SUM ( CASE WHEN B.Registro IS NULL THEN 1 ELSE 0 END ) 
                OVER( PARTITION BY A.Registro ORDER BY A.Data ) AS INT ) AS Total_dias,
            CAST ( COUNT(*)
                OVER( PARTITION BY A.Registro ORDER BY A.Data ) AS INT ) AS Validador,
            CAST ( COUNT(*)
                OVER( PARTITION BY A.Registro ORDER BY A.Data ) AS INT ) -     
                CAST ( SUM ( CASE WHEN B.Registro IS NULL THEN 1 ELSE 0 END ) 
                    OVER( PARTITION BY A.Registro ORDER BY A.Data ) AS INT ) AS Total_dias_afastados
    
    FROM
            contagem_acumulada AS A

    LEFT JOIN
            afastamentos_maior_15_dias AS B
        ON
            A.Registro = B.Registro
        AND
            A.Grupo = B.Grupo

""").df()

# Controle de atualização de processo: Etapa 4

linha_4 = [id, datetime.today(), 4]

ws_p.append(linha_4)

wb_p.save(path_registros_processos)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Marcação de Afastamento e Contagem de Dias concluído')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Marcação de Afastamento e Contagem de Dias concluído

----------------------------------------------------------------------------------------------------


# Bases com Marcação da Data de 45 Dias

In [33]:
# Base com os colaboradores únicos

registros_distintos = db.query("""

    SELECT DISTINCT
            Registro

    FROM
            contabiliza_periodo_experiencia

""").df()

# Base com os colaboradores que alcançaram ou alcançarão os 45 dias de experiência

Finalizaram_45_dias = db.query("""

    SELECT
            Registro,
            Data,
            CASE WHEN Total_dias <> Validador THEN 1 ELSE 0 END AS Teve_afastamento

    FROM
            contabiliza_periodo_experiencia

    WHERE
            Total_dias = 45 

""").df()

# Base com os colaboradores que não alcançaram ou alcançarão os 45 dias de experiência considerando afastamentos

Nao_finalizaram_45_dias = db.query("""

    WITH Nao_localizados AS (

        SELECT
                A.Registro
        
        FROM
                registros_distintos AS A
    
        LEFT JOIN        
                Finalizaram_45_dias AS B
            ON
                A.Registro = B.Registro

        WHERE
                B.Registro IS NULL
        
    )

    SELECT
            A.Registro,
            DATE_ADD( A.Data, INTERVAL ( 45 - Total_dias || ' day')) AS Data,
            1 AS Teve_afastamento            

    FROM
            contabiliza_periodo_experiencia AS A
            
    INNER JOIN
            Nao_localizados AS B
        ON
            A.Registro = B.Registro

    WHERE
            Data = CURRENT_DATE

""").df()

# Base consolidada de colaboradores e previsão de quando fizeram ou farão 45 dias de experiência

Base_45_dias_consolidado = db.query("""

    SELECT
            *

    FROM
            Finalizaram_45_dias

    UNION ALL

    SELECT
            *

    FROM
            Nao_finalizaram_45_dias

""").df()

# Controle de atualização de processo: Etapa 5

linha_5 = [id, datetime.today(), 5]

ws_p.append(linha_5)

wb_p.save(path_registros_processos)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Marcação da data de 45 dias concluído')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Marcação da data de 45 dias concluído

----------------------------------------------------------------------------------------------------


# Bases com Marcação da Data de 90 Dias

In [34]:
# Base com os colaboradores que alcançaram ou alcançarão os 90 dias de experiência

Finalizaram_90_dias = db.query("""

    SELECT
            Registro,
            Data,
            CASE WHEN Total_dias <> Validador THEN 1 ELSE 0 END AS Teve_afastamento

    FROM
            contabiliza_periodo_experiencia

    WHERE
            Total_dias = 90

""").df()

# Base com os colaboradores que não alcançaram ou alcançarão os 90 dias de experiência considerando afastamentos

Nao_finalizaram_90_dias = db.query("""

    WITH Nao_localizados AS (

        SELECT
                A.Registro
        
        FROM
                registros_distintos AS A
    
        LEFT JOIN        
                Finalizaram_90_dias AS B
            ON
                A.Registro = B.Registro

        WHERE
                B.Registro IS NULL
        
    )

    SELECT
            A.Registro,
            DATE_ADD( A.Data, INTERVAL ( 90 - Total_dias || ' day')) AS Data,
            1 AS Teve_afastamento            

    FROM
            contabiliza_periodo_experiencia AS A
            
    INNER JOIN
            Nao_localizados AS B
        ON
            A.Registro = B.Registro

    WHERE
            Data = CURRENT_DATE

""").df()

# Base consolidada de colaboradores e previsão de quando fizeram ou farão 90 dias de experiência

Base_90_dias_consolidado = db.query("""

    SELECT
            *

    FROM
            Finalizaram_90_dias

    UNION ALL

    SELECT
            *

    FROM
            Nao_finalizaram_90_dias

""").df()

# Controle de atualização de processo: Etapa 6

linha_6 = [id, datetime.today(), 6]

ws_p.append(linha_6)

wb_p.save(path_registros_processos)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Marcação da data de 90 dias concluído')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Marcação da data de 90 dias concluído

----------------------------------------------------------------------------------------------------


# Base Final

In [35]:
base_final = db.query("""

    SELECT
            A.Registro,
            B.Data                  AS Previsao_45,
            B.Teve_afastamento      AS Afastou_ate_45,
            C.Data                  AS Previsao_90,
            C.Teve_afastamento      AS Afastou_ate_90,
            CASE 
                WHEN A.Apto = 0 
                THEN 1 
                ELSE 0 
            END                     AS Afastado_atualmente,
            A.Total_dias_afastados  AS Total_dias_afastados

    FROM
            contabiliza_periodo_experiencia AS A

    LEFT JOIN
            Base_45_dias_consolidado AS B
        ON
            A.Registro = B.Registro

    LEFT JOIN
            Base_90_dias_consolidado AS C
        ON
            A.Registro = C.Registro

    WHERE
            A.Data = CURRENT_DATE

""").df()

# Controle de atualização de processo: Etapa 7

linha_7 = [id, datetime.today(), 7]

ws_p.append(linha_7)

wb_p.save(path_registros_processos)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Base final realizada')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Base final realizada

----------------------------------------------------------------------------------------------------


# Geração do Arquivo

In [36]:
# Diretório onde será salvo o arquivo final

caminho_arquivo_final = r'X:\Gestao_de_Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\PERIODO DE EXPERIÊNCIA.xlsx'

# Gerando o arquivo

wb = Workbook()
ws = wb.active
ws.title = "EXPERIENCIA"

for r in dataframe_to_rows(base_final, index=False, header=True):
    ws.append(r)

tabela_base_final = Table(displayName="EXPERIENCIA", ref=ws.dimensions)

estilo_tabela = TableStyleInfo(
    name="TableStyleLight13", 
    showFirstColumn=False,
    showLastColumn=False,
    showRowStripes=True,
    showColumnStripes=True
)

tabela_base_final.tableStyleInfo = estilo_tabela

ws.add_table(tabela_base_final)

wb.save(caminho_arquivo_final)

# Controle de atualização de processo: Etapa 8

linha_8 = [id, datetime.today(), 8]

ws_p.append(linha_8)

wb_p.save(path_registros_processos)

# RESUMO DE FINALIZAÇÃO DO PROCESSO

print('----------------------------------------------------------------------------------------------------')
print('')
print('   Processo finalizado')
print('')
print('   Tempo de execução:')
print('')
print(f'   {linha_8[1] - linha_0[1]}')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   Processo finalizado

   Tempo de execução:

   0:00:14.344094

----------------------------------------------------------------------------------------------------


# Atualização do Power BI - PERÍODO DE EXPERIÊNCIA

In [ ]:
tempo_0 = [id, datetime.today(), 0]

pyautogui.PAUSE = 1

# Entrar no Power BI
path_pbi = r"X:\Gestao_de_Pessoas\Analytics\09 - Power BI\PERÍODO DE EXPERIÊNCIA.pbix"
os.startfile(path_pbi) # Abre o arquivo pelo Windows
time.sleep(30)

# Atualizar Power BI
pyautogui.click(x=-830, y=103)
time.sleep(40)
pyautogui.click(x=-303, y=100)
time.sleep(5)
pyautogui.click(x=-745, y=473)
time.sleep(5)
pyautogui.click(x=-987, y=376)
pyautogui.press("tab")
pyautogui.press("enter")
time.sleep(5)
pyautogui.click(x=-758, y=575)
time.sleep(10)
pyautogui.click(x=-618, y=576)
time.sleep(5)
pyautogui.hotkey("alt", "F4")
time.sleep(2)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Power BI atualizado')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Power BI atualizado

----------------------------------------------------------------------------------------------------


In [ ]:
tempo_1 = [id, datetime.today(), 8]

# RESUMO DE FINALIZAÇÃO DO PROCESSO

print('----------------------------------------------------------------------------------------------------')
print('')
print('   Processo finalizado')
print('')
print('   Tempo de execução:')
print('')
print(f'   {tempo_1[1] - tempo_0[1]}')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   Processo finalizado

   Tempo de execução:

   0:02:21.399101

----------------------------------------------------------------------------------------------------
